[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/rl-lab/blob/main/01_openai_gym.ipynb)


# 01. Основы Gymnasium и случайный агент

Цель ноутбука — познакомиться с интерфейсом Gymnasium на среде CartPole-v1 и получить базовую линию качества для случайного агента.

**Результаты обучения:**
- создавать среду через `gym.make`;
- использовать современный API `reset` и `step`;
- отделять признаки завершения `terminated` и `truncated`;
- собирать статистику по эпизодам.

## Источник
Lapan M., *Deep Reinforcement Learning Hands-On*, главы 2-3.


In [ ]:
!pip install -q gymnasium


Подготовим библиотеки и зафиксируем генераторы случайных чисел. Блок с PyTorch оставлен общим для всех ноутбуков, даже если в этой части он не используется.


In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
except ImportError:
    pass


Создадим CartPole-v1 и посмотрим на пространства наблюдений и действий. Затем вручную сделаем несколько шагов, чтобы увидеть структуру кортежа, возвращаемого `step()`.


In [ ]:
env = gym.make("CartPole-v1")
obs, info = env.reset(seed=SEED)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Sample observation:", obs)
print("Sample action:", env.action_space.sample())

for step_idx in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print(step_idx, obs, reward, terminated, truncated, info)
    if terminated or truncated:
        obs, info = env.reset(seed=SEED + step_idx + 1)
env.close()


Теперь запустим случайного агента на 200 эпизодах. Для воспроизводимости каждый эпизод получает свой seed.


In [ ]:
env = gym.make("CartPole-v1")
rewards, lengths = [], []

for ep in range(200):
    obs, info = env.reset(seed=SEED + ep)
    total_reward, length = 0.0, 0
    done = False
    while not done:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        length += 1
        done = terminated or truncated
    rewards.append(total_reward)
    lengths.append(length)

env.close()
print(f"Mean reward: {np.mean(rewards):.2f}")
print(f"Max reward: {np.max(rewards):.0f}")


Гистограмма показывает, насколько нестабильна случайная политика и как редко она набирает большую награду.


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(rewards, bins=30, edgecolor="black")
plt.xlabel("Episode reward")
plt.ylabel("Count")
plt.title("Random agent reward distribution")
plt.grid(alpha=0.3)
plt.show()


Based on Lapan M., *Deep Reinforcement Learning Hands-On*, chapter 2-3.


In [ ]:
os.makedirs("results", exist_ok=True)
df_results = pd.DataFrame([
    ("mean_reward", np.mean(rewards)),
    ("std_reward", np.std(rewards)),
    ("min_reward", np.min(rewards)),
    ("max_reward", np.max(rewards)),
    ("frac_reward_ge_100", np.mean(np.array(rewards) >= 100)),
    ("frac_reward_eq_500", np.mean(np.array(rewards) == 500)),
], columns=["metric", "value"])
print(df_results.to_string(index=False))
df_results.to_csv("results/01_random_baseline.csv", index=False)
